In [ ]:
import mne
import numpy as np
import os
import shutil
import matplotlib.pyplot as plt

# se avete problemi a fare avviare le visualizzazioni potete provare con questo
# mne.viz.set_3d_backend('pyvistaqt')

# Set paths (You will need to adapt these for your class setup)
data_path = 'sub16ses2/eeg/'
subjects_dir = 'sub16ses2/'
subject = 'freesurfer' # Name of the freesurfer subject folder

# Resting state recording
## Caricare i dati
Carichiamo i dati di rest. Durante questa recording, il soggetto è rimasto fermo nello scanner a occhi chiusi per 10 minuti.

In [ ]:
# Load the Raw rest EEG data
raw = mne.io.read_raw_eeglab(data_path + 'sub-16_ses-02_task-rest_eeg.set', preload=True)

# Set the standard 10-20 montage for the 61 cortical channels
# only used when you don't have the real, digitized, positions
# we try it for now but we will remove it later
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

# settiamo il CAR
raw.set_eeg_reference('average', projection=True)
raw.apply_proj()

# CROP: Keep only 120 seconds of recording to speed up the processing later
# this recording has a few issues in the first minute, so we skip it
raw.crop(tmin=60.0, tmax=180.0)

# salva l'oggetto appena creato
raw.save('sub16ses2_raw_rest.fif', overwrite=True)

Ora esploriamo brevemente il dato. Qui vediamo i metadati:

In [ ]:
# Inspect the Metadata (The Info object)
# This is arguably the most important object in MNE. It contains your 
# sampling frequency, channel types, bad channels, and hardware info.
print(raw.info)

Ora visualizziamo la registrazione:

In [ ]:
# Interactive Time-Series Plot
# This will open an interactive window where you can scroll through time, 
# zoom in/out, and click on channels to mark them as "bad". 
# 'duration' sets the time window (seconds), 'n_channels' limits how many you see at once.
raw.plot(duration=5.0, n_channels=20, title="Continuous Data Explorer")

Ora guardiamo in frequenza:

In [ ]:
# Power Spectral Density (PSD)
# Let's look at the data in the frequency domain. This is great for spotting 
# alpha peaks (~10Hz) or checking if line noise (50/60Hz) was properly filtered.
# Note: In newer versions of MNE, compute_psd() is preferred over the old plot_psd().
raw.compute_psd(fmax=80).plot()

plt.show()

Questi dati sono già stati preprocessati, ma volendo potremmo eseguire tutto il preprocessing in MNE e salvare un nuovo file:

In [ ]:
######## NOT NEEDED, THIS DATA IS ALREADY PREPROCESSED

# if necessary, it should be done BEFORE average referencing

# # Apply a Notch filter at 60 Hz (US mains frequency)
# raw.notch_filter(freqs=60.0)
# # Filter the data (0.5 to 70 Hz as per the dataset's original paper)
# raw.filter(l_freq=0.5, h_freq=70.0)

# potete anche fare l'ICA, trovate le istruzioni qui:
# https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html

## Coregistrazione
Qui allineiamo le posizioni (nello spazio) degli elettrodi salvate dentro il file .set, alla superficie che abbiamo ricostruito dalla risonanza.

In [ ]:
## COREGISTRAZIONE AUTOMATICA

# Initialize the Coregistration object. 
# "estimated" tells MNE to guess the fiducial locations on the MRI based on standard anatomy.
coreg = mne.coreg.Coregistration(raw.info, subject=subject, subjects_dir=subjects_dir, fiducials="estimated")

# Initial alignment using just the 3 fiducial points
coreg.fit_fiducials(verbose=True)

# Refine the alignment using the Iterative Closest Point (ICP) algorithm.
# This mathematically rotates and translates the EEG cap until it best fits the curvature of the head.
coreg.fit_icp(n_iterations=20, nasion_weight=2.0, verbose=True)

# Save the transformation matrix to a file (not doing it now)
# coreg.trans.save('sub16ses2-trans.fif', overwrite=True)

# ---------------------------------------------------------
# VISUAL SANITY CHECK
# ---------------------------------------------------------
# We plot the head surface, the original electrode positions, and where they project onto the scalp.
# If the small spheres (electrodes) are floating too far above the gray head, the coregistration failed!
fig = mne.viz.plot_alignment(
    raw.info, 
    trans=coreg.trans, 
    subject=subject, 
    subjects_dir=subjects_dir, 
    surfaces='head-dense', # The high-res scalp surface
    show_axes=True, 
    dig=True,              # Show digitized points
    eeg=['original', 'projected'], # Show original and snapped positions
    coord_frame='mri'
)

In [ ]:
# avvia la coregistrazione manuale

mne.gui.coregistration(subject=subject, subjects_dir=subjects_dir, inst = 'sub16ses2_raw_rest.fif')

In [ ]:
# visualizza la coregistrazione
fig = mne.viz.plot_alignment(
    raw.info, 
    trans='sub16ses2-trans.fif', 
    subject=subject, 
    subjects_dir=subjects_dir, 
    surfaces='head-dense', # The high-res scalp surface
    show_axes=True, 
    dig=True,              # Show digitized points
    eeg=['original', 'projected'], # Show original and snapped positions
    coord_frame='mri'
)

## Definisci e risolvi il problema diretto


### crea lo spazio delle sorgenti

In [ ]:
# Setup the Source Space (the grid of dipoles on the cortex)
# add_dist='patch' speeds up the computations a bit now, but it's not recommended in general
# oct5 is a bit low, but you need more RAM as you go up in subdivisions
src = mne.setup_source_space(subject, spacing='oct5', add_dist='patch', subjects_dir=subjects_dir)

In [ ]:
# visualizza lo spazio delle sorgenti appena creato
src.plot(subjects_dir=subjects_dir)

In [ ]:
# maniera equivalente, ma più versatile, di visualizzare le sorgenti
fig = mne.viz.plot_alignment(
    subject=subject, 
    subjects_dir=subjects_dir, 
    surfaces='white',   # Shows the white matter surface
    coord_frame='mri',  # Plots in MRI coordinates
    src=src,            # Passes your source space object
    show_axes=True,
    interaction='terrain'
)

In [ ]:
src

Esiste anche la possibilità di creare uno spazio "volumetrico":

In [ ]:
vol_src = mne.setup_volume_source_space(
    subject,
    mri='aseg.mgz',
    pos=10.0,
    surface = subjects_dir+subject+'/bem/brain.surf',
    volume_label=None,
    subjects_dir=subjects_dir,
)


In [ ]:
vol_src.plot(subjects_dir=subjects_dir)

E uno misto (di solito usando settando volume_label in maniera da selezionare solo le strutture sottocorticali):

In [ ]:
# Generate the mixed source space
mixed_src = src + vol_src

In [ ]:
mixed_src.plot(subjects_dir=subjects_dir)

### Definisci le proprietà elettriche dei tessuti

In [ ]:
# Create the Boundary Element Model (BEM)
# This models how electrical currents travel through the brain, skull, and scalp.
model = mne.make_bem_model(subject=subject, conductivity=(0.3, 0.006, 0.3), subjects_dir=subjects_dir)
bem = mne.make_bem_solution(model)

### Crea la leadfield

In [ ]:
fwd = mne.make_forward_solution(raw.info, trans='sub16ses2-trans.fif', src=src, bem=bem, meg=False, eeg=True, n_jobs=-1)

In [ ]:
print(fwd.keys())

print(fwd['sol'])

## Risolvi il problema inverso

In [ ]:
# Compute noise covariance from the rest recording
noise_cov = mne.make_ad_hoc_cov(raw.info)

noise_cov.data

In [ ]:
# Assemble the inverse operator (loose and depth determine the amount of source constraint and depth-weighting)
inverse_operator = mne.minimum_norm.make_inverse_operator(raw.info, fwd, noise_cov, loose=0.2, depth=0.8)

In [ ]:
# Standard parameters (they SHOULD be tuned, but it's incredibly boring)
snr = 1.0
lambda2 = 1.0 / snr ** 2

stc = mne.minimum_norm.apply_inverse_raw(raw, inverse_operator, lambda2, method='MNE', pick_ori=None)

In [ ]:
# Plot the result on the 3D brain!
brain = stc.plot(subject=subject, subjects_dir=subjects_dir, initial_time=0.1, hemi='both', views='caudal', time_viewer=True, title=f"Source Space Activity")

In [ ]:
# we can also change a bit the parameters of the plot
brain = stc.plot(subject=subject, subjects_dir=subjects_dir, initial_time=0.1, hemi='lh', views='caudal', surface='pial', time_viewer=False, title=f"Source Space Activity")

### Your turn
#### Algoritmi di localizzazione delle sorgenti
Abbiamo appena usato *l'algoritmo MNE* per risolvere il problema inverso, ma **la libreria MNE** supporta anche altri metodi, prova a risolvere il problema inverso usandone un altro tra quelli disponibili:
- dSPM
- sLORETA
- eLORETA

#### Tipi di soluzione
Nell'applicare l'operatore inverso, abbiamo usato il parametro pick_ori=None, dalla documentazione di MNE:
- pick_ori: None | “normal” | “vector”
    - **None**: Pooling is performed by taking the norm of loose/free orientations. In case of a fixed source space no norm is computed leading to signed source activity.
    - **"normal"**: Only the normal to the cortical surface is kept. This is only implemented when working with loose orientations.
    - **"vector"**: No pooling of the orientations is done, and the vector result will be returned in the form of a mne.VectorSourceEstimate object.

Scegli un metodo tra quelli di sopra, e prova a cambiare la variabile pick_ori, cosa cambia?

### Algoritimi di localizzazione
Visualizzare la soluzione in questo modo non ci aiuta a capire quali metodi funzionano meglio. Inoltre, alcuni di questi metodi sono pensati per mitigare il "bias superficiale" dell'algoritmo MNE. Facciamo un esempio di risoluzione volumetrica del problema inverso con sLORETA.

In [ ]:
# create a new leadfield on the volume source space
vol_fwd = mne.make_forward_solution(raw.info, trans='sub16ses2-trans.fif', src=vol_src, bem=bem, meg=False, eeg=True, n_jobs=-1)

# create the inverse operator with the new leadfield
inverse_operator = mne.minimum_norm.make_inverse_operator(raw.info, vol_fwd, noise_cov, loose='auto')

In [ ]:
snr = 1.0
lambda2 = 1.0 / snr ** 2

# solve the inverse problem using MNE and sLORETA
stc_MNE = mne.minimum_norm.apply_inverse_raw(raw, inverse_operator, lambda2, method='MNE', pick_ori=None)
stc_sLORETA = mne.minimum_norm.apply_inverse_raw(raw, inverse_operator, lambda2, method='sLORETA', pick_ori=None)

In [ ]:
%matplotlib qt
stc_MNE.plot(subject=subject, subjects_dir=subjects_dir, initial_time=22.5, src = vol_src);

# Checker task recording
Ora carichiamo i dati del task, e dividiamo la registrazione in chunk da 2 secondi con i label task e rest.

In [ ]:
# Load the Raw EEG data
checkerOut = mne.io.read_raw_eeglab(data_path + 'sub-16_ses-02_task-checkerout_eeg.set', preload=True)

In [ ]:
# settiamo il CAR
checkerOut.set_eeg_reference('average', projection=True)
checkerOut.apply_proj()

In [ ]:
# esploriamo il task
checkerOut.plot(duration=5.0, n_channels=20, title="Continuous Data Explorer")

In [ ]:
# Extract events (Replace '1' and '2' with your actual trigger codes for Task and Rest)
print(mne.events_from_annotations(checkerOut))
events, event_dict = mne.events_from_annotations(checkerOut)
rest_id, task_id = event_dict['S 12'], event_dict['S 27']

# Create Epochs
epochs_task = mne.Epochs(checkerOut, events, event_id=task_id, tmin=0, tmax=2, baseline=None, preload=True)
epochs_rest = mne.Epochs(checkerOut, events, event_id=rest_id, tmin=0, tmax=2, baseline=None, preload=True)

## Definisci e risolvi il problema diretto
Lo spazio delle sorgenti e il modello elettrico dipendono solo dal soggetto, quindi rimangono le stesse che abbiamo calcolato sopra.

In [ ]:
src, model, bem

### Crea la leadfield
La leadfield dipende da quali elettrodi abbiamo in questa registrazione e da dove sono, quindi, in linea di principio, andrebbe ricalcolata per registrazioni fatte in giornate diverse (quindi con possibilità di elettrodi che vengono aggiunti o rimossi per problemi tecnici). Questo vale anche per la coregistrazione (ma noi la salteremo); dobbiamo rigenerare però la forward solution.

In [ ]:
# Calculate the Forward Solution
# Note: 'trans.fif' is the coregistration matrix aligning the EEG cap to the MRI. We are using the same we computed with the rest recording
fwd = mne.make_forward_solution(checkerOut.info, trans='sub16ses2-trans.fif', src=src, bem=bem, meg=False, eeg=True, n_jobs=-1)

## Risolvi il problema inverso

In [ ]:
# create covariance matrix
noise_cov = mne.make_ad_hoc_cov(checkerOut.info)

# Assemble the inverse operator (loose and depth determine the amount of source constraint and depth-weighting)
inverse_operator = mne.minimum_norm.make_inverse_operator(checkerOut.info, fwd, noise_cov, loose=0.2, depth=0.8)

### Singola epoca
Abbiamo svariati trial, concentriamoci sul primo e vediamo se riusciamo a estrarre qualcosa di utile.

In [ ]:
snr = 1.0 
lambda2 = 1.0 / snr ** 2

# let's fix a specific method and focus on the results
method = 'MNE'

# We pass just the first epoch (epochs_task_chopped[:1]) so it runs instantly
stc_single = mne.minimum_norm.apply_inverse_epochs(
    epochs_task[:1], 
    inverse_operator, 
    lambda2, 
    method=method, 
    return_generator=False,
    pick_ori = None
)[0]

In [ ]:
# Plot the single-trial time course
brain_single = stc_single.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    surface='white',
    initial_time=0.1,
    hemi='both',
    views='caudal',
    time_viewer=True,
    title='Single Epoch Time Course'
)

L'andamento temporale non è molto informativo, proviamo con il PSD direttamente nello spazio delle sorgenti.

In [ ]:
stc_single.data

In [ ]:
# Compute PSD on the raw data array of the source estimate
psds, freqs = mne.time_frequency.psd_array_multitaper(
    stc_single.data, 
    sfreq=stc_single.sfreq, 
    n_jobs=-1
)

psds, freqs

In [ ]:
def plot_psd_stc(psds, freqs, stc):
    # we now have the psd for each point in our source space, but how do we plot it?
    # There is no built-in method to do this, so we create a fake source time course object
    # and trick MNE into thinking that frequency is actually time

    # 1. Create a dummy STC using the spatial info from earlier
    psd_single = stc.copy()

    # 2. Calculate the frequency resolution (our fake 'tstep')
    # freqs is an array like [1.0, 1.5, 2.0, ...], so the step is 0.5
    freq_step = freqs[1] - freqs[0]

    # 3. Inject the PSD data and hack the time axis
    psd_single.tmin = freqs[0]       # Set start time to starting frequency (1.0 Hz)
    psd_single.tstep = freq_step     # Set step size to frequency resolution
    psd_single.data = psds  # Swap voltage for Power!

    # 3. Plot the Interactive Frequency Brain
    brain_single = psd_single.plot(
        subject=subject,
        subjects_dir=subjects_dir,
        surface='white',
        hemi='both',
        views='caudal',
        time_label='Frequency (Hz)', # Rename the label in the viewer
        time_viewer=True,            # Turns on the magical frequency slider!
    )
    
    return brain_single

In [ ]:
plot_psd_stc(psds, freqs, stc_single)

### Averaging!
Non si vede molto sul singolo trial, ma abbiamo raccolto molti trial durante l'esperimento, quindi possiamo mediare per aumentare il SNR!

Mediare nel tempo distruggerebbe il segnale SSVEP che vogliamo osservare, quindi calcoliamo il psd di tutti i trial e solo dopo facciamo la media.

In [ ]:
snr = 1.0 
lambda2 = 1.0 / snr ** 2

# let's fix a method and focus on the results
method = 'MNE'

# We pass all the epochs now
stc_task = mne.minimum_norm.apply_inverse_epochs(
    epochs_task, 
    inverse_operator, 
    lambda2, 
    method=method, 
    return_generator=True,
    pick_ori = 'vector'
)

# we iterate through each epoch to compute the psd of each
# the frequencies should stay the same, since the epochs are all the same length
psds_task = []
for stc in stc_task:
    psds, freqs = mne.time_frequency.psd_array_multitaper(
        stc.data, 
        sfreq=stc.sfreq, 
        n_jobs=-1
    )
    
    psds_task.append(psds)

psds_task = np.stack(psds_task, axis = 0)
psds_task = np.linalg.norm(psds_task, axis=-2)
avg_psd_task=np.mean(psds_task, axis = 0)
std_psd_task=np.std(psds_task, axis = 0)

avg_psd_task, freqs

In [ ]:
plot_psd_stc(avg_psd_task, freqs, stc_single)

### Your turn!
Ora fai la stessa cosa con i trial di rest e calcola un avg_psd_rest e std_psd_rest corrispondenti.

### Contrasto!
Nel segnale mediato vediamo ancora delle attivazioni che sono difficili da interpretare, per capire se sono collegate al task o se sono attività spontanea possiamo fare un contrasto con il rest.

In [ ]:
D = (std_psd_rest**2 / psds_rest.shape[0]) + (std_psd_task**2 / psds_task.shape[0])

# Prevent division by zero
D[np.isclose(D, 0)] = 1.0

zstat_spectrum = (avg_psd_task - avg_psd_rest) / np.sqrt(D)

plot_psd_stc(zstat_spectrum, freqs, stc_single)